In [ ]:
import os
import numpy as np
import pandas as pd

import pickle
from pathlib import Path

import os
import subprocess

In [ ]:
data = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-11-07_viral_disease_nonseasonal_case_cohort_binning"
scratch = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/scratch/2025-11-07_viral_disease_nonseasonal_case_cohort_binning"


results1 = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-11-05_viral_disease_cohort" 

In [ ]:
#mapping specie:vaccine list

def mapping_vax_list_to_concept():
    
    annot_vax_cohort = pd.read_csv(f"{scratch}/annot_vax_cohort.csv")
    
    vaccine_concepts = pd.read_csv(f'{data}/ns_viral_cohort_vax_ids.csv')
    
    vaccine_concept_df = vaccine_concepts.loc[:, ['updated_specie', 'vaccine_id']]
    
    
    ns_cohort_vaccine_df = annot_vax_cohort.merge(vaccine_concept_df,  on='updated_specie', how='left')
    
    
    
    return ns_cohort_vaccine_df

In [ ]:
def get_concept_vax_id_dict(ns_cohort_vax_df): 
    '''
       1. subset a new df with only 2 columns: concept_id and vaccine_id
       2. reset the index to concept_id
       3. groupby concept_id, selecting (vaccine_id) col as a series, and using .agg to apply an aggregation to each 
        group, that being the lambda function to make the grouped series into a single list
    
    ''' 
    
    df1 = ns_cohort_vax_df.loc[ :, ['condition_concept_id', 'vaccine_id']]
    df2 = df1.set_index('condition_concept_id')
    df3 = df2.groupby('condition_concept_id', sort=False)['vaccine_id'].agg(lambda vax_id: list(vax_id))
    final = df3.to_dict()
    
    
    return final

In [ ]:
def get_vaccine_table(vaccine_ids):
    """
    Fetches drug exposure rows for the specified vaccine_concept_id(s),
    applying Cohort Builder UI filters (EHR + genomics, flat‐events,
    standard concepts), and returns a DataFrame with:
      - person_id
      - drug_concept_id
      - standard_concept_name
      - drug_exposure_start/end_datetime
      - verbatim_end_date
      - source_concept_name
    """
    dataset = os.environ["WORKSPACE_CDR"]
    # allow passing either a single int or a list/tuple of ints
    if not isinstance(vaccine_ids, (list, tuple)):
        vaccine_ids = [vaccine_ids]
    ids_sql = "(" + ",".join(str(i) for i in vaccine_ids) + ")"

    sql = f"""
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name AS drug_standard_concept_name,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_source_concept.concept_name AS source_concept_name 
    FROM (
        SELECT * 
        FROM `{dataset}.drug_exposure` d_exposure 
        WHERE
            drug_concept_id IN (
                SELECT DISTINCT ca.descendant_id 
                FROM `{dataset}.cb_criteria_ancestor` ca 
                JOIN (
                    SELECT DISTINCT c.concept_id       
                    FROM `{dataset}.cb_criteria` c       
                    JOIN (
                        SELECT CAST(cr.id AS STRING) AS id             
                        FROM `{dataset}.cb_criteria` cr             
                        WHERE
                            cr.concept_id IN {ids_sql}           
                            AND cr.full_text LIKE '%_rank1]%'       
                    ) a 
                      ON (
                        c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id
                      ) 
                    WHERE
                        c.is_standard   = 1 
                        AND c.is_selectable = 1
                ) b 
                  ON ca.ancestor_id = b.concept_id
            )
          AND d_exposure.person_id IN (
            SELECT DISTINCT p.person_id  
            FROM `{dataset}.cb_search_person` p  
            WHERE
                p.has_ehr_data = 1 
              AND (
                   p.has_whole_genome_variant    = 1 
                OR p.has_lr_whole_genome_variant = 1 
                OR p.has_array_data              = 1 
              )
          )
          AND d_exposure.person_id IN (
            SELECT criteria.person_id 
            FROM (
              SELECT DISTINCT person_id, entry_date, concept_id 
              FROM `{dataset}.cb_search_all_events` 
              WHERE
                concept_id IN (
                  SELECT DISTINCT c.concept_id 
                  FROM `{dataset}.cb_criteria` c 
                  JOIN (
                    SELECT CAST(cr.id AS STRING) AS id       
                    FROM `{dataset}.cb_criteria` cr       
                    WHERE
                        cr.concept_id IN (440029)       
                      AND cr.full_text LIKE '%_rank1]%'      
                  ) a 
                    ON (
                      c.path LIKE CONCAT('%.', a.id, '.%') 
                      OR c.path LIKE CONCAT('%.', a.id) 
                      OR c.path LIKE CONCAT(a.id, '.%') 
                      OR c.path = a.id
                    ) 
                  WHERE
                    c.is_standard   = 1 
                    AND c.is_selectable = 1
                )
                AND is_standard = 1
            ) criteria
          )
    ) d_exposure 
    LEFT JOIN `{dataset}.concept` d_standard_concept 
      ON d_exposure.drug_concept_id       = d_standard_concept.concept_id 
    LEFT JOIN `{dataset}.concept` d_source_concept 
      ON d_exposure.drug_source_concept_id = d_source_concept.concept_id
    """

    df = pd.read_gbq(
        sql,
        project_id=os.environ.get("BIGQUERY_PROJECT"),
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )
    return df

In [ ]:

annot_vax_cohort = pd.read_csv(f"{scratch}/annot_vax_cohort.csv")

COND_NAME_MAP = dict(
    zip(annot_vax_cohort['condition_concept_id'], annot_vax_cohort['standard_concept_name']))



def get_cond_vax_table(ns_vax_id_dict): 
    
    ns_vax_cohort_dict = {}
    
    
    for key, vax_list in ns_vax_id_dict.items() :
    
        vax = get_vaccine_table(vax_list)
        
  
        #find dictionary keys, and add final tables to dict
        cond_name = COND_NAME_MAP.get(key, "<unknown>")
        ns_vax_cohort_dict[(key, cond_name)] = vax
    

    return ns_vax_cohort_dict

In [ ]:
#
def create_df_pkl(df, directory):

    # 2) Choose a workspace folder for persistence
    out_file = Path(directory)

    # 3) Save the entire dict in one go
    with open(out_file, 'wb') as f:
        pickle.dump(df, f)

    print(f"Saved {len(df)} DataFrames to {out_file}")

In [ ]:
# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

def export_pkl_to_WS_bucket(file_name):
    


    # Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
    destination_filename = file_name

    ########################################################################
    ##
    ################# DON'T CHANGE FROM HERE ###############################
    ##
    ########################################################################



    # get the bucket name
    my_bucket = os.getenv('WORKSPACE_BUCKET')

    # copy csv file to the bucket
    args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
    output = subprocess.run(args, capture_output=True)

    # print output from gsutil
    output.stderr


In [ ]:
#vax_map = mapping_vax_list_to_concept()
#concept_vax_id_dict = get_concept_vax_id_dict(vax_map)
#vax_df_dict = get_cond_vax_table(concept_vax_id_dict)

create_df_pkl(vax_df_dict, f"{data}/viral_disease_cohort_vaccine_df_dict.pkl")
export_pkl_to_WS_bucket('viral_disease_cohort_vaccine_df_dict.pkl')

In [ ]:
##visualization 

#annot_vax_cohort 

#vax_map

#concept_vax_id_dict

#vax_df_dict


